<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/15A_NeuroFHIR_Review_WISH_Separate_GitHub_Pages_Deployment_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-Review — Notebook 15A
## Separate GitHub Pages Deployment for the Frozen 12-Case WISH Reviewer App

**Purpose:** publish the already validated NeuroFHIR-Review participant application at a **separate URL** without changing or replacing the existing NeuroFHIR-QC / AMIA application.

### Target URL
`https://sanghati23.github.io/neurofhir-qc/wish-review/`

### What this notebook does
- reads **only** the frozen participant application created by the final WISH pilot build;
- refuses to copy the `researcher_only` package;
- scans the deployment for researcher-only leakage markers;
- clones a fresh copy of the GitHub repository;
- stages the app under `app/frontend/public/wish-review/`;
- builds the existing frontend and verifies that `dist/wish-review/index.html` is produced;
- runs a local HTTP smoke test;
- creates a deployment manifest with SHA-256 hashes;
- optionally commits and pushes the deployment to `main`;
- optionally waits for GitHub Pages and verifies the new public URL.

**This notebook does not change the 12-case study logic, counterbalancing, case content, or reviewer instrumentation.**
It deploys the frozen participant-facing build as-is.

## Study sequence

Current project sequence:

`13 build → 14 case expansion → 14B final stimulus/counterbalance validation → 14C end-to-end dry run → 15 reviewer training/rubric → 15A separate participant-app deployment → Step 9 real recruitment/data collection → 16 real pilot analysis`

`P001` and `P002` remain **QA/dry-run IDs only**. Real study participants should begin with `P003`.

In [21]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
# Cell 2 — Configuration
from pathlib import Path

GITHUB_OWNER = "SANGHATI23"
GITHUB_REPO = "neurofhir-qc"
GITHUB_BRANCH = "main"

DRIVE_REPO_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
WISH_ROOT = DRIVE_REPO_ROOT / "wish_extension"

# Frozen participant application produced by the validated final pilot package.
FROZEN_PARTICIPANT_APP = (
    WISH_ROOT / "final_wish_pilot" / "participant_app"
)

# The researcher-only directory is intentionally separate and MUST NEVER be published.
RESEARCHER_ONLY_DIR = (
    WISH_ROOT / "final_wish_pilot" / "researcher_only"
)

# Fresh deployment checkout.
CLONE_ROOT = Path("/content/neurofhir-qc_wish_deploy")
FRONTEND_ROOT = CLONE_ROOT / "app" / "frontend"
PUBLIC_WISH_DIR = FRONTEND_ROOT / "public" / "wish-review"
DIST_WISH_DIR = FRONTEND_ROOT / "dist" / "wish-review"

PUBLIC_URL = (
    f"https://{GITHUB_OWNER.lower()}.github.io/"
    f"{GITHUB_REPO}/wish-review/"
)

# Keep False for the first validation run.
# After ALL gates are TRUE, change this to True and rerun from the clone/publish cells.
PUSH_TO_GITHUB = True

# Optional: after push, wait for the public URL to update and verify it.
VERIFY_PUBLIC_URL_AFTER_PUSH = True

print("Frozen participant app:", FROZEN_PARTICIPANT_APP)
print("Target public URL:", PUBLIC_URL)
print("Push enabled:", PUSH_TO_GITHUB)

Frozen participant app: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app
Target public URL: https://sanghati23.github.io/neurofhir-qc/wish-review/
Push enabled: True


In [23]:
# Cell 3 — Validate that the frozen participant package exists
import os

assert DRIVE_REPO_ROOT.exists(), f"Drive project root not found: {DRIVE_REPO_ROOT}"
assert FROZEN_PARTICIPANT_APP.exists(), (
    f"Frozen participant app not found: {FROZEN_PARTICIPANT_APP}"
)
assert (FROZEN_PARTICIPANT_APP / "index.html").exists(), (
    "Expected participant_app/index.html was not found."
)

print("✅ Frozen participant application found.")
print("Participant files:")
for p in sorted(FROZEN_PARTICIPANT_APP.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(FROZEN_PARTICIPANT_APP))

✅ Frozen participant application found.
Participant files:
 - assets/WISH-01.png
 - assets/WISH-02.png
 - assets/WISH-03.png
 - assets/WISH-04.png
 - assets/WISH-05.png
 - assets/WISH-06.png
 - assets/WISH-07.png
 - assets/WISH-08.png
 - assets/WISH-09.png
 - assets/WISH-10.png
 - assets/WISH-11.png
 - assets/WISH-12.png
 - index.html
 - participant_cases.json


In [24]:
# Cell 4 — Frozen-app integrity and participant/researcher separation gate
import re
from pathlib import Path

# Researcher-only variables/labels that should never be present in a participant deployment.
FORBIDDEN_TEXT_MARKERS = [
    "source_case_id",
    "ai_correctness",
    "reference_workflow_disposition_path_b",
    "path_A_reference_status",
    "final_researcher_scenario_key",
    "researcher_only",
    "researcher trajectory",
    "reference_volume",
    "reference volume",
]

FORBIDDEN_FILENAME_MARKERS = [
    "researcher",
    "answer_key",
    "scenario_key",
    "reference_workflow",
]

TEXT_SUFFIXES = {
    ".html", ".htm", ".js", ".mjs", ".cjs", ".json",
    ".css", ".txt", ".csv", ".md", ".xml"
}

leaks = []

for p in FROZEN_PARTICIPANT_APP.rglob("*"):
    if not p.is_file():
        continue

    rel = str(p.relative_to(FROZEN_PARTICIPANT_APP)).lower()

    for marker in FORBIDDEN_FILENAME_MARKERS:
        if marker.lower() in rel:
            leaks.append(f"filename:{rel} -> {marker}")

    if p.suffix.lower() in TEXT_SUFFIXES:
        try:
            txt = p.read_text(encoding="utf-8", errors="ignore").lower()
        except Exception:
            continue
        for marker in FORBIDDEN_TEXT_MARKERS:
            if marker.lower() in txt:
                leaks.append(f"content:{rel} -> {marker}")

assert not leaks, (
    "❌ PARTICIPANT/RESEARCHER SEPARATION FAILED. "
    "Do not deploy. Potential leakage:\n" + "\n".join(leaks[:50])
)

# Explicitly ensure researcher-only material is NOT nested inside participant_app.
assert not any(
    "researcher" in str(p.relative_to(FROZEN_PARTICIPANT_APP)).lower()
    for p in FROZEN_PARTICIPANT_APP.rglob("*")
), "Researcher-labelled material found inside participant app."

print("✅ Participant/researcher separation gate: PASS")
print("Researcher-only directory remains outside deployment:", RESEARCHER_ONLY_DIR)

✅ Participant/researcher separation gate: PASS
Researcher-only directory remains outside deployment: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/researcher_only


In [25]:
# Cell 5 — Confirm the app looks like the frozen NeuroFHIR-Review study UI
index_text = (FROZEN_PARTICIPANT_APP / "index.html").read_text(
    encoding="utf-8", errors="ignore"
)

# These are intentionally broad because wording may differ slightly in the frozen app.
index_lower = index_text.lower()

expected_groups = {
    "study identity": ["neurofhir-review", "neurofhir review"],
    "initial judgment": ["initial judgment"],
    "AI stage": ["ai recommendation", "ai review"],
    "final action": ["final action"],
}

missing_groups = []
for label, alternatives in expected_groups.items():
    if not any(term in index_lower for term in alternatives):
        missing_groups.append(label)

assert not missing_groups, (
    "The frozen index.html does not contain expected study UI markers: "
    + ", ".join(missing_groups)
)

print("✅ Frozen NeuroFHIR-Review UI identity gate: PASS")

✅ Frozen NeuroFHIR-Review UI identity gate: PASS


In [26]:
# Cell 6 — Hash the frozen source before deployment
import hashlib, json, datetime

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

source_hashes = {}
for p in sorted(FROZEN_PARTICIPANT_APP.rglob("*")):
    if p.is_file():
        rel = p.relative_to(FROZEN_PARTICIPANT_APP).as_posix()
        source_hashes[rel] = sha256_file(p)

print(f"✅ Hashed {len(source_hashes)} frozen participant files.")
print("index.html SHA-256:", source_hashes.get("index.html"))

✅ Hashed 14 frozen participant files.
index.html SHA-256: 0600d1ad3fb6706fcc322dc6407d4b8ab54a3acf7cd176b2b833178bc604c4c3


In [27]:
# Cell 7 — Fresh clone of the repository
import shutil, subprocess, os, sys

if CLONE_ROOT.exists():
    shutil.rmtree(CLONE_ROOT)

repo_url = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"

subprocess.run(
    ["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch", repo_url, str(CLONE_ROOT)],
    check=True,
)

assert (CLONE_ROOT / ".git").exists()
assert FRONTEND_ROOT.exists(), f"Frontend root not found: {FRONTEND_ROOT}"
assert (FRONTEND_ROOT / "package.json").exists(), (
    f"Expected frontend package.json not found: {FRONTEND_ROOT / 'package.json'}"
)

print("✅ Fresh repository clone complete.")

✅ Fresh repository clone complete.


In [28]:
# Cell 8 — Inspect deployment structure without altering the existing AMIA app
import json

package_json = json.loads((FRONTEND_ROOT / "package.json").read_text())

print("Frontend:", FRONTEND_ROOT)
print("Available npm scripts:")
for k, v in package_json.get("scripts", {}).items():
    print(f" - {k}: {v}")

print("\nExisting root app files remain untouched.")
print("New participant study path will be:")
print(PUBLIC_WISH_DIR)

Frontend: /content/neurofhir-qc_wish_deploy/app/frontend
Available npm scripts:
 - dev: vite --host 0.0.0.0
 - build: tsc -b && vite build
 - preview: vite preview --host 0.0.0.0

Existing root app files remain untouched.
New participant study path will be:
/content/neurofhir-qc_wish_deploy/app/frontend/public/wish-review


In [29]:
# Cell 9 — Copy ONLY the frozen participant app into the separate public subdirectory
import shutil

if PUBLIC_WISH_DIR.exists():
    shutil.rmtree(PUBLIC_WISH_DIR)

PUBLIC_WISH_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(FROZEN_PARTICIPANT_APP, PUBLIC_WISH_DIR)

assert (PUBLIC_WISH_DIR / "index.html").exists()

print("✅ Frozen participant app copied to:")
print(PUBLIC_WISH_DIR)
print("\nNo files were copied from:")
print(RESEARCHER_ONLY_DIR)

✅ Frozen participant app copied to:
/content/neurofhir-qc_wish_deploy/app/frontend/public/wish-review

No files were copied from:
/content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/researcher_only


In [30]:
# Cell 10 — Byte-for-byte freeze verification after copy
deployed_source_hashes = {}
for p in sorted(PUBLIC_WISH_DIR.rglob("*")):
    if p.is_file():
        rel = p.relative_to(PUBLIC_WISH_DIR).as_posix()
        deployed_source_hashes[rel] = sha256_file(p)

assert source_hashes == deployed_source_hashes, (
    "❌ Deployment staging changed the frozen participant files."
)

print("✅ Byte-for-byte frozen-copy gate: PASS")

✅ Byte-for-byte frozen-copy gate: PASS


In [31]:
# Cell 11 — URL/path safety scan
# The frozen app should be portable under /neurofhir-qc/wish-review/.
# Relative asset URLs are ideal. Absolute site-root references can break a subpath deployment.

import re

html = (PUBLIC_WISH_DIR / "index.html").read_text(encoding="utf-8", errors="ignore")

# Ignore protocol URLs such as https:// and data: URLs.
absolute_local_refs = re.findall(
    r'''(?:src|href)\s*=\s*["']/(?!/)\S*?["']''',
    html,
    flags=re.IGNORECASE,
)

assert not absolute_local_refs, (
    "❌ Found site-root absolute asset links in frozen index.html. "
    "Do not silently modify the frozen study build. "
    "Rebuild/freeze the participant app for subpath-safe relative assets first.\n"
    + "\n".join(absolute_local_refs[:20])
)

print("✅ Subpath portability scan: PASS")

✅ Subpath portability scan: PASS


In [32]:
# Cell 12 — Build the EXISTING frontend
# Vite/React public assets should be copied into dist during the normal build.
# This cell proves that the existing root application still builds AND that
# wish-review survives the build as a separate route.

import subprocess, os, shutil

npm_ci = ["npm", "ci"] if (FRONTEND_ROOT / "package-lock.json").exists() else ["npm", "install"]

subprocess.run(npm_ci, cwd=FRONTEND_ROOT, check=True)

scripts = package_json.get("scripts", {})
assert "build" in scripts, "No npm 'build' script found."

subprocess.run(["npm", "run", "build"], cwd=FRONTEND_ROOT, check=True)

assert (FRONTEND_ROOT / "dist" / "index.html").exists(), (
    "Existing root AMIA/NeuroFHIR-QC app was not built."
)
assert (DIST_WISH_DIR / "index.html").exists(), (
    "wish-review/index.html did not survive the frontend build. "
    "STOP: the current build pipeline is not copying public/wish-review."
)

print("✅ Frontend build gate: PASS")
print("✅ Existing root app output present:", FRONTEND_ROOT / "dist" / "index.html")
print("✅ Separate WISH app output present:", DIST_WISH_DIR / "index.html")

✅ Frontend build gate: PASS
✅ Existing root app output present: /content/neurofhir-qc_wish_deploy/app/frontend/dist/index.html
✅ Separate WISH app output present: /content/neurofhir-qc_wish_deploy/app/frontend/dist/wish-review/index.html


In [33]:
# Cell 13 — Verify built WISH app still matches the frozen source
built_hashes = {}
for p in sorted(DIST_WISH_DIR.rglob("*")):
    if p.is_file():
        rel = p.relative_to(DIST_WISH_DIR).as_posix()
        built_hashes[rel] = sha256_file(p)

assert source_hashes == built_hashes, (
    "❌ Build output for wish-review is not byte-for-byte identical "
    "to the frozen participant package."
)

print("✅ Frozen build-output gate: PASS")

✅ Frozen build-output gate: PASS


In [34]:
# Cell 14 — Local HTTP smoke test of BOTH routes
import subprocess, time, requests, socket, os, signal

server = subprocess.Popen(
    ["python", "-m", "http.server", "8765", "--directory", str(FRONTEND_ROOT / "dist")],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

try:
    time.sleep(2)

    root_resp = requests.get("http://127.0.0.1:8765/", timeout=10)
    wish_resp = requests.get("http://127.0.0.1:8765/wish-review/", timeout=10)

    assert root_resp.status_code == 200, f"Root app HTTP {root_resp.status_code}"
    assert wish_resp.status_code == 200, f"WISH app HTTP {wish_resp.status_code}"

    assert (
        "NeuroFHIR-Review" in wish_resp.text
        or "NeuroFHIR Review" in wish_resp.text
    ), "WISH route responded but did not look like NeuroFHIR-Review."

    print("✅ Local root-app smoke test: PASS")
    print("✅ Local /wish-review/ smoke test: PASS")
finally:
    server.terminate()
    server.wait(timeout=10)

✅ Local root-app smoke test: PASS
✅ Local /wish-review/ smoke test: PASS


In [35]:
# Cell 15 — Create deployment manifest
import datetime, json

manifest = {
    "artifact": "NeuroFHIR-Review frozen participant application",
    "deployment_subpath": "/wish-review/",
    "target_url": PUBLIC_URL,
    "source_directory": str(FROZEN_PARTICIPANT_APP),
    "source_file_count": len(source_hashes),
    "source_sha256": source_hashes,
    "researcher_only_deployed": False,
    "qa_participant_ids_reserved": ["P001", "P002"],
    "recommended_first_real_participant_id": "P003",
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
}

manifest_path = PUBLIC_WISH_DIR / "deployment_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("✅ Deployment manifest written:")
print(manifest_path)
print("\nNOTE: manifest contains hashes/configuration only — no researcher answer key.")

✅ Deployment manifest written:
/content/neurofhir-qc_wish_deploy/app/frontend/public/wish-review/deployment_manifest.json

NOTE: manifest contains hashes/configuration only — no researcher answer key.


In [36]:
# Cell 16 — Rebuild after adding the public deployment manifest
# The manifest is intentionally a deployment artifact, not part of the frozen study logic.

subprocess.run(["npm", "run", "build"], cwd=FRONTEND_ROOT, check=True)

assert (DIST_WISH_DIR / "index.html").exists()
assert (DIST_WISH_DIR / "deployment_manifest.json").exists()

# Frozen participant files must still be identical.
for rel, expected_hash in source_hashes.items():
    built_file = DIST_WISH_DIR / rel
    assert built_file.exists(), f"Frozen participant file missing after rebuild: {rel}"
    assert sha256_file(built_file) == expected_hash, f"Frozen participant file changed: {rel}"

print("✅ Final build gate: PASS")

✅ Final build gate: PASS


In [37]:
# Cell 17 — Git scope gate: only /public/wish-review may be COMMITTED
#
# This cell supports BOTH situations:
#   1) first deployment: wish-review files are untracked
#   2) redeployment: wish-review files are already tracked and now modified
#
# npm/Vite build artifacts are ignored because Cell 18 stages only:
#   app/frontend/public/wish-review/

import subprocess

INTENDED_PREFIX = "app/frontend/public/wish-review/"

# A) Tracked changes (modified/deleted files already known to git)
tracked_changes = subprocess.run(
    ["git", "diff", "--name-only"],
    cwd=CLONE_ROOT,
    text=True,
    capture_output=True,
    check=True,
).stdout.strip().splitlines()

# B) Untracked files
untracked = subprocess.run(
    ["git", "ls-files", "--others", "--exclude-standard"],
    cwd=CLONE_ROOT,
    text=True,
    capture_output=True,
    check=True,
).stdout.strip().splitlines()

# C) Expected local build artifacts from npm / tsc / Vite.
def is_expected_generated(path: str) -> bool:
    return (
        path.startswith("app/frontend/node_modules/")
        or path.startswith("app/frontend/dist/")
        or path in {
            "app/frontend/tsconfig.app.tsbuildinfo",
            "app/frontend/tsconfig.node.tsbuildinfo",
            "app/frontend/vite.config.js",
            "app/frontend/vite.config.d.ts",
        }
    )

# No TRACKED source file outside wish-review may have changed.
tracked_bad = [
    path for path in tracked_changes
    if path and not path.startswith(INTENDED_PREFIX)
]

assert not tracked_bad, (
    "❌ Tracked files outside the intended WISH deployment subtree changed:\n"
    + "\n".join(tracked_bad)
)

# No unexpected UNTRACKED file outside wish-review / known build outputs.
unexpected_untracked = [
    path for path in untracked
    if path
    and not path.startswith(INTENDED_PREFIX)
    and not is_expected_generated(path)
]

assert not unexpected_untracked, (
    "❌ Unexpected untracked files were created outside the WISH deployment subtree:\n"
    + "\n".join(unexpected_untracked)
)

# Detect actual deployment changes in either tracked OR untracked wish-review files.
wish_tracked_changes = [
    path for path in tracked_changes
    if path.startswith(INTENDED_PREFIX)
]

wish_untracked_changes = [
    path for path in untracked
    if path.startswith(INTENDED_PREFIX)
]

wish_changes = sorted(set(wish_tracked_changes + wish_untracked_changes))

assert wish_changes, (
    "❌ Git detects no changes under app/frontend/public/wish-review/.\n"
    "If you just ran Notebook 15A1, verify that its facelift was promoted to:\n"
    "  /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app/index.html\n"
    "and then rerun Cells 7–17 of Notebook 15A."
)

generated = [
    path for path in untracked
    if is_expected_generated(path)
]

print("✅ Git scope gate: PASS")
print(f"Tracked wish-review changes:   {len(wish_tracked_changes)}")
print(f"Untracked wish-review changes: {len(wish_untracked_changes)}")
print(f"Total deployment changes:      {len(wish_changes)}")

if generated:
    print(
        f"ℹ️ Ignoring {len(generated)} local npm/Vite build artifacts "
        "(Cell 18 will not stage them)."
    )

print("\nFiles eligible for this deployment commit:")
for path in wish_changes[:30]:
    print(" -", path)

if len(wish_changes) > 30:
    print(f" ... plus {len(wish_changes) - 30} more")

print("\nOnly this subtree may be staged:")
print(INTENDED_PREFIX)

✅ Git scope gate: PASS
Tracked wish-review changes:   2
Untracked wish-review changes: 0
Total deployment changes:      2
ℹ️ Ignoring 5975 local npm/Vite build artifacts (Cell 18 will not stage them).

Files eligible for this deployment commit:
 - app/frontend/public/wish-review/deployment_manifest.json
 - app/frontend/public/wish-review/index.html

Only this subtree may be staged:
app/frontend/public/wish-review/


In [38]:
# Cell 18 — Commit + push to GitHub main (only when explicitly enabled)
import os, subprocess, tempfile, stat

if not PUSH_TO_GITHUB:
    print("🛑 PUSH_TO_GITHUB=False")
    print("Validation is complete, but nothing was pushed.")
    print("When ready, set PUSH_TO_GITHUB=True in Cell 2 and rerun Cells 7–18.")
else:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
    assert token, (
        "Colab Secret GITHUB_TOKEN was not found. "
        "Add it in Colab → Secrets, grant notebook access, then rerun."
    )

    subprocess.run(
        ["git", "config", "user.name", "Sanghati Basu"],
        cwd=CLONE_ROOT,
        check=True,
    )
    subprocess.run(
        ["git", "config", "user.email", "sanghati@users.noreply.github.com"],
        cwd=CLONE_ROOT,
        check=True,
    )

    subprocess.run(
        ["git", "add", "app/frontend/public/wish-review"],
        cwd=CLONE_ROOT,
        check=True,
    )

    staged = subprocess.run(
        ["git", "diff", "--cached", "--name-only"],
        cwd=CLONE_ROOT,
        text=True,
        capture_output=True,
        check=True,
    ).stdout.strip().splitlines()

    assert staged, "Nothing staged for commit."
    assert all(
        p.startswith("app/frontend/public/wish-review/")
        for p in staged
    ), "Unexpected file staged outside wish-review."

    commit_msg = "Deploy frozen NeuroFHIR-Review participant app at /wish-review/"
    commit = subprocess.run(
        ["git", "commit", "-m", commit_msg],
        cwd=CLONE_ROOT,
        text=True,
        capture_output=True,
    )

    if commit.returncode not in (0, 1):
        print(commit.stdout)
        print(commit.stderr)
        raise RuntimeError("git commit failed")

    # Use GIT_ASKPASS so the token is not placed in the remote URL or printed.
    askpass = Path("/tmp/neurofhir_git_askpass.sh")
    askpass.write_text(
        '''#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
  *) echo "" ;;
esac
''',
        encoding="utf-8",
    )
    askpass.chmod(0o700)

    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GITHUB_TOKEN"] = token
    env["GIT_TERMINAL_PROMPT"] = "0"

    push = subprocess.run(
        ["git", "push", "origin", f"HEAD:{GITHUB_BRANCH}"],
        cwd=CLONE_ROOT,
        env=env,
        text=True,
        capture_output=True,
    )

    # Remove askpass immediately.
    try:
        askpass.unlink()
    except FileNotFoundError:
        pass

    if push.returncode != 0:
        print(push.stdout)
        print(push.stderr)
        raise RuntimeError("GitHub push failed.")

    print("✅ GitHub push complete.")
    print("Expected Pages URL:", PUBLIC_URL)

✅ GitHub push complete.
Expected Pages URL: https://sanghati23.github.io/neurofhir-qc/wish-review/


In [39]:
# Cell 19 — Optional public GitHub Pages verification
import time, requests

if not PUSH_TO_GITHUB:
    print("Skipped because nothing was pushed.")
elif not VERIFY_PUBLIC_URL_AFTER_PUSH:
    print("Skipped because VERIFY_PUBLIC_URL_AFTER_PUSH=False.")
else:
    print("Waiting for GitHub Pages deployment...")
    deadline = time.time() + 10 * 60
    last_status = None
    verified = False

    # deployment_manifest.json is unique to this deployment and helps avoid
    # accidentally accepting the old root app as success.
    public_manifest_url = PUBLIC_URL + "deployment_manifest.json"

    while time.time() < deadline:
        try:
            r = requests.get(PUBLIC_URL, timeout=15, headers={"Cache-Control": "no-cache"})
            m = requests.get(public_manifest_url, timeout=15)
            last_status = (r.status_code, m.status_code)

            if (
                r.status_code == 200
                and m.status_code == 200
                and (
                    "NeuroFHIR-Review" in r.text
                    or "NeuroFHIR Review" in r.text
                )
            ):
                verified = True
                break
        except Exception as e:
            last_status = repr(e)

        time.sleep(20)

    assert verified, (
        "GitHub push succeeded, but the new Pages URL was not verified "
        f"within 10 minutes. Last result: {last_status}. "
        "Check the GitHub Actions Pages workflow before changing anything."
    )

    print("✅ PUBLIC DEPLOYMENT VERIFIED")
    print(PUBLIC_URL)

Waiting for GitHub Pages deployment...
✅ PUBLIC DEPLOYMENT VERIFIED
https://sanghati23.github.io/neurofhir-qc/wish-review/


In [40]:
# Cell 20 — Final deployment gate
final_gate = (
    (FROZEN_PARTICIPANT_APP / "index.html").exists()
    and (PUBLIC_WISH_DIR / "index.html").exists()
    and (DIST_WISH_DIR / "index.html").exists()
)

print("=" * 72)
print("NOTEBOOK 15A — SEPARATE WISH DEPLOYMENT GATE")
print("=" * 72)
print("Frozen participant source:", "PASS")
print("Researcher-only separation:", "PASS")
print("Byte-for-byte frozen copy:", "PASS")
print("Subpath portability:", "PASS")
print("Existing frontend build:", "PASS")
print("Separate /wish-review/ build:", "PASS")
print("Local HTTP smoke test:", "PASS")
print("Git change scope:", "PASS")
print("Push requested:", PUSH_TO_GITHUB)
print("Target URL:", PUBLIC_URL)
print("=" * 72)

if PUSH_TO_GITHUB:
    print("✅ NOTEBOOK 15A DEPLOYMENT GATE: TRUE")
    print("The frozen participant app should now be available at:")
    print(PUBLIC_URL)
else:
    print("✅ NOTEBOOK 15A PRE-PUBLISH VALIDATION GATE: TRUE")
    print("Nothing has been pushed yet.")
    print("Set PUSH_TO_GITHUB=True only after reviewing all PASS gates.")

NOTEBOOK 15A — SEPARATE WISH DEPLOYMENT GATE
Frozen participant source: PASS
Researcher-only separation: PASS
Byte-for-byte frozen copy: PASS
Subpath portability: PASS
Existing frontend build: PASS
Separate /wish-review/ build: PASS
Local HTTP smoke test: PASS
Git change scope: PASS
Push requested: True
Target URL: https://sanghati23.github.io/neurofhir-qc/wish-review/
✅ NOTEBOOK 15A DEPLOYMENT GATE: TRUE
The frozen participant app should now be available at:
https://sanghati23.github.io/neurofhir-qc/wish-review/
